In [10]:
import pandas as pd
import numpy as np

In [11]:
# Load database
df_database = pd.read_csv('../data/car_database.csv', sep=';')

In [ ]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# --- AHP CONFIG (Subjective Criteria) ---
ahp_criteria_direction = {
    'cost': 'min', 
    'horsepower': 'max', 
    'transmission': 'max', 
    'city_fuel_economy': 'max', 
    'ground_clearance': 'max', 
    'power_windows': 'max',
    'power_side_mirrors': 'max', 
    'infotainment_system': 'max', 
    'rear_parking_sensors': 'max', 
    'fog_lights': 'max', 
    'roof_rails': 'max',
    'aesthetics_index': 'max',
    'freshness_index': 'max'
}

# Hierarchy Tiers
tier_0_essential    = ['cost', 'freshness_index']
tier_1_must_have    = ['infotainment_system', 'rear_parking_sensors', 'fog_lights', 'freshness_index']
tier_2_very_nice    = ['horsepower', 'ground_clearance',  'roof_rails', 'aesthetics_index']
tier_3_nice_to_have = ['transmission', 'power_windows', 'power_side_mirrors', 'city_fuel_economy']

# --- GAUSSIAN CONFIG (Objective Criteria) ---
gaussian_candidates = [
    'cost',                 
    'torque',               
    'highway_fuel_economy', 
    'payload_capacity',
    'freshness_index'
]

# --- LOAD DATABASE ---
try:
    df_database = pd.read_csv('../data/car_database.csv', sep=';')
except FileNotFoundError:
    print("File not found. Please check the path.")
    df_database = pd.DataFrame()

# ==============================================================================
# 2. PROCESSING SUBJECTIVE COLUMNS
# ==============================================================================

def process_subjective_columns(df, raw_col, target_index_col, intensity=0.25):
    """
    Reads a raw score column (1-5 scale) and converts to normalized index.
    Formula: Index = 1 + ((RawScore - 3) / 2) * Intensity
    """
    df_proc = df.copy()
    if raw_col in df_proc.columns:
        df_proc[raw_col] = pd.to_numeric(df_proc[raw_col], errors='coerce').fillna(3)
        df_proc[target_index_col] = 1 + ((df_proc[raw_col] - 3) / 2) * intensity
    else:
        # Fallback
        # print(f"Warning: {raw_col} missing. Using neutral index.")
        df_proc[target_index_col] = 1.0
    return df_proc

# Apply Subjective Processing
if not df_database.empty:
    # Aesthetics (Visual)
    df_database = process_subjective_columns(df_database, 'raw_aesthetics', 'aesthetics_index', intensity=0.25)
    # Freshness (How new the project is)
    df_database = process_subjective_columns(df_database, 'raw_freshness', 'freshness_index', intensity=0.25)

# ==============================================================================
# 3. DUAL PRE-PROCESSING (YOUR CUSTOM RULES + SPLIT PIPELINE)
# ==============================================================================

def create_dual_dataframes(df):
    
    # --- A. Common Cleaning (Applying YOUR specific rules) ---
    df_common = df.copy()
    
    # 1. Transmission
    if 'transmission' in df_common.columns:
        df_common['transmission'] = df_common['transmission'].apply(lambda x: 1 if str(x).lower().strip() == 'automatic' else 0)
    
    # 2. Ground Clearance
    if 'ground_clearance' in df_common.columns:
        df_common['ground_clearance'] = df_common['ground_clearance'].apply(
            lambda x: 2 if str(x).lower().strip() == 'high' else (1 if str(x).lower().strip() == 'medium' else 0)
        )

    # 3. Air Conditioning (Digital preference)
    if 'air_conditioning' in df_common.columns:
        df_common['air_conditioning'] = df_common['air_conditioning'].apply(lambda x: 1 if str(x).lower().strip() == 'digital' else 0)

    # 4. Steering Wheel Adjustment (Tilt & Telescopic)
    if 'steering_wheel_adjustment' in df_common.columns:
        df_common['steering_wheel_adjustment'] = df_common['steering_wheel_adjustment'].apply(
            lambda x: 2 if str(x).lower().strip() == 'tilt and telescopic' else (1 if str(x).lower().strip() == 'tilt' else 0)
        )

    # 5. Power Steering
    if 'power_steering' in df_common.columns:
        df_common['power_steering'] = df_common['power_steering'].apply(lambda x: 1 if str(x).lower().strip() == 'electric' else 0)

    # 6. Power Windows (Front & Rear)
    if 'power_windows' in df_common.columns:
        df_common['power_windows'] = df_common['power_windows'].apply(
            lambda x: 2 if str(x).lower().strip() == 'front and rear' else (1 if str(x).lower().strip() == 'front' else 0)
        )

    # 7. General Type Conversion
    for col in df_common.columns:
        if df_common[col].dtype == 'bool': df_common[col] = df_common[col].astype(int)
        if df_common[col].dtype == 'object': df_common[col] = pd.to_numeric(df_common[col], errors='coerce')

    if 'car' in df_common.columns and 'version' in df_common.columns:
        df_common['car_version'] = df_common['car'] + ' - ' + df_common['version']
        df_common.set_index('car_version', inplace=True)
    
    df_numeric = df_common.select_dtypes(include=[np.number])

    # --- B. Create DF AHP (HARD LOGIC) ---
    # Keeps the 0, 1, 2 values exactly as you defined above
    df_ahp_hard = df_numeric.copy()

    # --- C. Create DF GAUSSIAN (SOFT LOGIC) ---
    # Applies damping to booleans to reduce artificial variance
    df_gauss_soft = df_numeric.copy()
    
    # Soft Booleans: We dampen strict booleans (0/1) for the Gaussian engine
    soft_boolean_cols = ['rear_parking_sensors', 'fog_lights', 'roof_rails']
    penalty = 0.10 
    
    for col in soft_boolean_cols:
        if col in df_gauss_soft.columns:
            # Check if it is binary (0/1) before applying soft logic
            unique_vals = df_gauss_soft[col].unique()
            if len(unique_vals) <= 2 and df_gauss_soft[col].max() == 1:
                df_gauss_soft[col] = df_gauss_soft[col].apply(lambda x: 1.0 if x == 1 else (1.0 - penalty))
            
    return df_ahp_hard, df_gauss_soft

# Generate
if not df_database.empty:
    df_ahp, df_gauss = create_dual_dataframes(df_database)
else:
    df_ahp, df_gauss = pd.DataFrame(), pd.DataFrame()

# ==============================================================================
# 4. AHP MODULE (SUBJECTIVE)
# ==============================================================================

def calculate_ahp_weights(criteria_dict, t0, t1, t2, t3):
    cols = list(criteria_dict.keys())
    n = len(cols)
    if n == 0: return pd.Series()
    
    matrix = np.ones((n, n))
    idx = {name: i for i, name in enumerate(cols)}
    
    STEP_ESSENTIAL_MUST = 2.0
    STEP_MUST_VERYNICE = 2.0
    STEP_VERYNICE_NICE = 2.0 
    
    def apply_weight(strong, weak, weight):
        for s in strong:
            for w in weak:
                if s in idx and w in idx:
                    matrix[idx[s], idx[w]] = weight
                    matrix[idx[w], idx[s]] = 1.0 / weight

    apply_weight(t0, t1, STEP_ESSENTIAL_MUST)
    apply_weight(t1, t2, STEP_MUST_VERYNICE)
    apply_weight(t2, t3, STEP_VERYNICE_NICE)
    apply_weight(t0, t2, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE)
    apply_weight(t1, t3, STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)
    apply_weight(t0, t3, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)
    
    col_sums = matrix.sum(axis=0)
    weights = (matrix / col_sums).mean(axis=1)
    return pd.Series(weights, index=cols)

weights_ahp_raw = calculate_ahp_weights(
    ahp_criteria_direction, 
    tier_0_essential, tier_1_must_have, tier_2_very_nice, tier_3_nice_to_have
)

print("--- AHP Weights ---")
print(weights_ahp_raw.sort_values(ascending=False).head(15))
print("-" * 30)

# ==============================================================================
# 5. GAUSSIAN MODULE (OBJECTIVE)
# ==============================================================================

def calculate_gaussian_weights(df, whitelist_cols):
    available_cols = [c for c in whitelist_cols if c in df.columns]
    valid_cols = []
    for c in available_cols:
        if df[c].std() > 0:
            valid_cols.append(c)
    if not valid_cols: return pd.Series()

    df_g = df[valid_cols].copy()
    norm_matrix = df_g / df_g.sum()
    gaussian_factors = norm_matrix.std() / norm_matrix.mean()
    weights = gaussian_factors / gaussian_factors.sum()
    return weights

# Using SOFT dataframe for stability
weights_gauss_raw = calculate_gaussian_weights(df_gauss, whitelist_cols=gaussian_candidates)

print("--- Gaussian Weights ---")
print(weights_gauss_raw.sort_values(ascending=False).head(5))
print("-" * 30)

# ==============================================================================
# 6. SMART HYBRID ENGINE (DUAL PIPELINE)
# ==============================================================================

def execute_smart_hybrid_decision_dual(df_hard, df_soft, w_ahp, w_gauss, ahp_directions):
    
    def normalize(df_target, weights):
        df_n = pd.DataFrame(index=df_target.index)
        cols = [c for c in weights.index if c in df_target.columns]
        for col in cols:
            direction = ahp_directions.get(col, 'max')
            vals = df_target[col]
            if vals.sum() == 0: df_n[col] = 0; continue
            if direction == 'min':
                inv = 1 / (vals + 1e-9)
                df_n[col] = inv / inv.sum()
            else:
                df_n[col] = vals / vals.sum()
        return df_n

    norm_ahp = normalize(df_hard, w_ahp)
    norm_gauss = normalize(df_soft, w_gauss)

    # Calculate Independent Scores
    cols_ahp = [c for c in w_ahp.index if c in norm_ahp.columns]
    score_ahp_only = norm_ahp[cols_ahp].dot(w_ahp[cols_ahp])
    
    cols_gauss = [c for c in w_gauss.index if c in norm_gauss.columns]
    score_gauss_only = norm_gauss[cols_gauss].dot(w_gauss[cols_gauss])
    
    # Measure Dispersion
    std_ahp = score_ahp_only.std()
    std_gauss = score_gauss_only.std()
    
    total_variation = std_ahp + std_gauss
    smart_ahp_share = 0.5 if total_variation == 0 else std_ahp / total_variation
    
    print(f"\n--- Smart Weighting Calculated ---")
    print(f"AHP Share: {smart_ahp_share*100:.2f}% | Gaussian Share: {(1-smart_ahp_share)*100:.2f}%")
    print("-" * 30)

    # Final Score
    df_results = pd.DataFrame(index=df_hard.index)
    df_results['Score_AHP_Part'] = score_ahp_only
    df_results['Score_Gauss_Part'] = score_gauss_only
    df_results['Score_Final'] = (score_ahp_only * smart_ahp_share) + (score_gauss_only * (1 - smart_ahp_share))
    
    return df_results.sort_values('Score_Final', ascending=False)

# Execute
if not df_ahp.empty:
    ranking = execute_smart_hybrid_decision_dual(
        df_ahp, df_gauss, weights_ahp_raw, weights_gauss_raw, ahp_criteria_direction
    )

    print("\n=== FINAL RANKING ===")
    cols_view = ['cost', 'aesthetics_index', 'freshness_index', 'Score_Final'] 
    cols_view = [c for c in cols_view if c in ranking.columns]
    print(ranking[cols_view].head(10))

--- AHP Weights ---
cost                    0.222222
fog_lights              0.111111
freshness_index         0.111111
infotainment_system     0.111111
rear_parking_sensors    0.111111
roof_rails              0.055556
aesthetics_index        0.055556
ground_clearance        0.055556
horsepower              0.055556
city_fuel_economy       0.027778
transmission            0.027778
power_windows           0.027778
power_side_mirrors      0.027778
dtype: float64
------------------------------
--- Gaussian Weights ---
torque                  0.575869
cost                    0.207968
highway_fuel_economy    0.157931
payload_capacity        0.058232
dtype: float64
------------------------------

--- Smart Weighting Calculated ---
AHP Share: 71.72% | Gaussian Share: 28.28%
------------------------------

=== FINAL RANKING ===
                                                    Score_Final
car_version                                                    
Citroën C3 2026  - 1.0 TURBO 200 FLEX YOU